### Set up

In [66]:
import json
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)

<module 'src.translate_to_vis' from 'c:\\Users\\bhatt\\Desktop\\UCIrvine\\RelCogLab_work\\graph_extract\\src\\translate_to_vis.py'>

In [67]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
# set main paths
CUR_DIR = os.path.dirname(os.path.abspath(__name__))
# SCENARIO_DIR = CUR_DIR+'/scenarios/'
SCENARIO_DIR = CUR_DIR+'/formatted_franken/data/conditions_mild_harm_mild_good/'
DATA_DIR_HUMAN = CUR_DIR+'/data/human_annotation/'
OUTPUT_DIR = CUR_DIR+'/franken_annotated_outputs/'

## Look at Scenario

In [69]:
#set scenario file filename
FILENAME = 'cc_evitable_prevention_no_stories.json'

#select scenario and action choice
SCENARIO_ID = 1
ACT_ID = '1'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


I, an Engineer, face a moral dilemma. My colleague has the opportunity to upgrade the company's
server system. As a means to slightly improve data processing speed, the upgrade process requires
the company's server system to be offline for a few hours, causing a temporary halt in the company's
operations. If I prevent his colleague's server system upgrade, there would not be a temporary halt
in the company's operations. 




In [70]:
# print scenario json
print(json.dumps(scenario_json, indent=4))

{
    "id": 1,
    "text": "I, an Engineer, face a moral dilemma. My colleague has the opportunity to upgrade the company's server system. As a means to slightly improve data processing speed, the upgrade process requires the company's server system to be offline for a few hours, causing a temporary halt in the company's operations. If I prevent his colleague's server system upgrade, there would not be a temporary halt in the company's operations.",
    "options": {
        "1": "I prevent his colleague's server system upgrade.",
        "2": "I do not prevent his colleague's server system upgrade."
    }
}


#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

#### Set up

In [71]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
# this_act_I = "I decide to " + this_act
this_act_I = this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
print(scenario_json['text'])
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Ziv prevents his colleague's server system upgrade.
I prevent his colleague's server system upgrade.
I, an Engineer, face a moral dilemma. My colleague has the opportunity to upgrade the company's server system. As a means to slightly improve data processing speed, the upgrade process requires the company's server system to be offline for a few hours, causing a temporary halt in the company's operations. If I prevent his colleague's server system upgrade, there would not be a temporary halt in the company's operations.


In [72]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')

Graph g initialized and reset.


#### Step 0. Get entities

In [73]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]

#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list





Identified these entities: 

I
my colleague


#### Step 1. Deontology / Action Value Scores

In [74]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
processed_values  = annotate_scenario.process_value_simple(this_act,this_act_I,g) 

print(processed_values)

    

You are an expert judge of the moral virtues of different actions. Moral actions are those which humans consider virtuous, that consider others' wellbeing and happiness, and that are guided by principles of ethics. You will be given a sentence describing an action. You must rate thus action in terms of its moral virtue. Respond with a rating from -100 to 100 for this action, where -100 is the most morally wrong action you can possibly think of, and 100 is the most morally virtuous action you can possibly think of. 0 indicates a morally neutral action. Please return a json object with key:value pair of "score": rating. Please rate this action: I prevent his colleague's server system upgrade.
{'score': -20}


#### Step 2. Outcome Likelihoods

In [75]:
#Step 2. Outcomes

processed_events = annotate_scenario.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

The company's server system remains online without interruption
The company's operations continue without a temporary halt
The data processing speed does not improve
The potential benefits of the server upgrade are not realized
My colleague's plan to upgrade the server is blocked


#### Step 3. Outcome Utilities

In [76]:
#Step 3. Outcome utilities

impacts_list = annotate_scenario.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: The company's server system remains online without interruption
Scored impacts for these beings:
['Ziv', "Ziv's colleague"]
Scored values:
[20, -30]


IndexError: list index out of range

#### Step 4. Cause / Intend / Know Links

In [ ]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: The company's server system remains online without interruption
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The company's operations continue without a temporary halt
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The data processing speed remains at the current level
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: No immediate improvement in system performance occurs
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+


#### Step 5. Write out the results

In [ ]:
#optional -- write out the results 

this_output_filename = f"{OUTPUT_DIR}scenarios_{SCENARIO_ID}_choice_{ACT_ID}.json"
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename,g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: c:\Users\bhatt\Desktop\UCIrvine\RelCogLab_work\graph_extract/franken_annotated_outputs/scenarios_1_choice_2.json



c:\Users\bhatt\Desktop\UCIrvine\RelCogLab_work\graph_extract/franken_annotated_outputs/scenarios_1_choice_2.json
